[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/two-phaseflow/JED116F/blob/main/notebooks/03_stored_heat_and_fractured_rock.ipynb)

Click the badge to open this notebook in Google Colab. In Colab, choose **File > Save a copy in Drive** before you start, so your work is kept.

# JEÐ116F Notebook 03. Stored heat, Monte Carlo, and heat extraction from fractured rock

Lecture 8, Thursday 29 October 2026. Parts 1 and 2 are done in class in two blocks, Parts 3 and 4 together in a third block after the fracture model has been built on the board. The completed notebook is part 2 of Assignment 3 (part 1 is notebook 02, the well tests); the last section says what to hand in.

**Cell types**

- **A. Run and look.** The code is complete. Run it and read the output.
- **B. Fill the gaps.** Replace every `___` with a number or a short expression. The expected values are given in the text so you can check yourself.
- **Possible extensions.** Open questions for when you finish early.

**Rules for this course**

1. Keep **units** next to every number in your code comments.
2. Nothing physical is typed in by hand when a property package can give it: water properties come from IAPWS-IF97 (`iapws`).
3. You may use AI tools for error messages, plotting and coding ideas. You must be able to explain every line you hand in.

**Parts**

1. Stored heat and electrical capacity: the equations in code
2. Monte Carlo: the answer as a range, and which input controls it
3. Heat extraction from parallel fractures: the Gringarten model in code, run and explored
4. The recovery factor computed, not assumed
5. The same model for a doublet (Fervo type), the tool for Assignment 3, part 2

**Assignment 3, part 2** at the end: what to hand in.

**Symbols used throughout**

| symbol | meaning | unit |
|---|---|---|
| $Q$ | heat stored in the block above $T_0$ | J |
| $\rho c$ | volumetric heat capacity of the rock | J/(m³ K) |
| $A,\ h,\ V = A h$ | area, thickness and volume of the block | m², m, m³ |
| $T,\ T_0$ | reservoir temperature, reference temperature | °C |
| $P$ | electrical capacity | W (we print MWe) |
| $R$ | recovery factor: thermal energy produced at the wellhead over the project life, above $T_0$, divided by the thermal energy in place above $T_0$ | |
| $\eta$ | conversion efficiency, electrical over thermal at the wellhead | |
| $F$ | capacity factor, fraction of the time at full load | |
| $t$ | plant life | s |
| $\dot m$ | total mass flow through the fractured block | kg/s |
| $N,\ H,\ L,\ 2x_E$ | number, height, width and spacing of the fractures | m |
| $m' = \dot m/(N L)$ | mass flow per fracture per metre of width | kg/(m s) |
| $c_w$ | specific heat of water | J/(kg K) |
| $\lambda,\ \rho_R c_R,\ \kappa = \lambda/(\rho_R c_R)$ | rock conductivity, volumetric heat capacity, thermal diffusivity | W/(m K), J/(m³ K), m²/s |
| $T_{R0},\ T_{in},\ T_{out}$ | initial rock, injection and outlet temperature | °C |
| $\ell = 2 \lambda H/(c_w m')$ | Gringarten's length scale; $t_1 = \ell^2/\kappa$ is the time scale of the cooling | m |

## Part 0. Setup

No data files are needed. Water properties come from the IAPWS-IF97 formulation in the `iapws` package.

In [ ]:
%pip install -q iapws

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp     # an ODE solver, used in Part 3
from scipy.stats import spearmanr         # rank correlation, used in Part 2
from iapws import IAPWS97

plt.rcParams.update({"font.size": 13, "figure.figsize": (8, 5), "axes.grid": True})
YEAR = 365.25 * 86400.0                    # s

def u_water(T_C, P_MPa=15.0):
    '''Specific internal energy of liquid water, J/kg, from IAPWS-IF97. 15 MPa keeps the water liquid up to 340 C.'''
    return IAPWS97(P=P_MPa, T=T_C + 273.15).u * 1e3

def rho_water(T_C, P_MPa=15.0):
    '''Density of liquid water, kg/m3, from IAPWS-IF97.'''
    return IAPWS97(P=P_MPa, T=T_C + 273.15).rho

def cw_mean(T1_C, T2_C, P_MPa=15.0):
    '''Mean specific heat of liquid water between two temperatures, J/(kg K): enthalpy difference over temperature difference.'''
    h1 = IAPWS97(P=P_MPa, T=T1_C + 273.15).h * 1e3
    h2 = IAPWS97(P=P_MPa, T=T2_C + 273.15).h * 1e3
    return (h2 - h1) / (T2_C - T1_C)

print(f"u(300 C) - u(30 C) = {(u_water(300) - u_water(30))/1e3:.0f} kJ/kg;  mean c_w between 40 and 300 C = {cw_mean(40, 300):.0f} J/(kg K)")

## Part 1. Stored heat and electrical capacity

The stored heat method counts the heat in a block of rock above a reference temperature $T_0$ and converts a fraction of it into electricity over the life of the plant:

$$ Q = \rho c \, A h \, (T - T_0) \qquad\qquad P = \frac{Q R \eta}{F t} $$

$Q$ stored heat (J); $\rho c$ volumetric heat capacity of the rock (J/(m³ K)); $A$ area (m²); $h$ thickness (m); $T$ reservoir temperature and $T_0$ reference temperature (°C); $P$ electrical capacity (W); $R$ recovery factor; $\eta$ conversion efficiency; $F$ capacity factor; $t$ plant life (s).

### B1.1 The two equations as functions

Fill the two gaps. Check: one cubic kilometre at 250 °C with $T_0$ = 30 °C and $\rho c$ = 2.6 MJ/(m³ K) holds 5.7 × 10¹⁷ J, and with $R$ = 0.10, $\eta$ = 0.10, $F$ = 0.9 and 30 years gives 6.7 MWe.

In [ ]:
def stored_heat(A, h, T, T0=30.0, rho_c=2.6e6):
    '''Heat stored in the rock of a block above T0, J.  A m2, h m, T and T0 C, rho_c J/(m3 K).'''
    return ___   # the stored heat equation

def capacity(Q, R=0.10, eta=0.10, F=0.9, life_years=30.0):
    '''Electrical capacity, W, from the stored heat Q (J) over a plant life in years.'''
    return ___   # the capacity equation

Q_km3 = stored_heat(A=1e6, h=1e3, T=250.0)           # one cubic kilometre: 1 km2 times 1 km
P_km3 = capacity(Q_km3)
print(f"Q = {Q_km3:.2e} J,  P = {P_km3/1e6:.1f} MWe for 30 years")
assert abs(Q_km3 - 5.72e17) < 2e16 and abs(P_km3 / 1e6 - 6.7) < 0.2

### B1.2 How much of the heat is in the water, and how much in the rock?

Each cubic metre of reservoir is $(1 - \phi)$ cubic metres of rock and $\phi$ cubic metres of water. The rock holds $(1 - \phi)\, \rho_R c_R (T - T_0)$ and the pore water holds $\phi\, \rho_w (u(T) - u(T_0))$, where $u$ is the specific internal energy of liquid water from IAPWS (internal energy rather than enthalpy, because the water in the reservoir is not flowing; Zarrouk and Simiyu 2013). Fill the water term: porosity times the water density at $T$ times the change in internal energy from $T_0$ to $T$, with the helpers `rho_water(T)` and `u_water(T)` from Part 0, so `phi * rho_water(T) * (u_water(T) - u_water(T0))`. Read the share of the heat that is in the water for porosities of 2, 5, 10 and 20 percent. Expected: about 10 percent of the heat is in the water at $\phi = 0.08$.

Why this matters: whatever the porosity of a real reservoir, most of the heat is in the rock. Getting heat out of rock is the subject of the second half of this notebook.

In [ ]:
T, T0 = 250.0, 30.0
rho_c_R = 2.6e6                                   # J/(m3 K), rock
for phi in (0.02, 0.05, 0.08, 0.10, 0.20):
    q_rock = (1 - phi) * rho_c_R * (T - T0)       # J per m3 of reservoir, in the rock
    q_water = ___   # J per m3 of reservoir, in the pore water (density at T)
    share = q_water / (q_rock + q_water)
    print(f"porosity {phi:4.2f}:  rock {q_rock:.2e} J/m3,  water {q_water:.2e} J/m3,  share in the water {share*100:4.0f} %")
assert abs(q_water / (q_rock + q_water) - 0.22) < 0.05      # the 20 percent case
print("The rock-only equation (Ciriaco et al. 2020, equation 8) misses this share; at the porosity of a real reservoir it is 5 to 10 percent.")

### A. The reference temperature and the recovery factor go together

Keep $R$ = 0.10 and move $T_0$ from 30 to 150 °C. $P$ falls because less of the heat is counted. In practice a report that uses the plant rejection temperature as $T_0$ uses a higher $R$ (Grant 2018: 13 ± 5 percent with ambient $T_0$, 27 ± 15 percent with separator $T_0$, for the same fields).

In [ ]:
for T0 in (30.0, 100.0, 150.0):
    P = capacity(stored_heat(1e6, 1e3, 250.0, T0=T0))
    print(f"T0 = {T0:5.0f} C:  P = {P/1e6:.1f} MWe at R = 0.10")

## Part 2. Monte Carlo: the answer as a range

Before drilling, every input is a range. The Monte Carlo method draws one random value of each input, computes $P$, and repeats many times. The result is a distribution of $P$, reported as P90, P50 and P10: the values exceeded with 90, 50 and 10 percent probability.

Two shapes of distribution are used here. **Uniform**: any value in the range is equally likely. **Triangular**: the middle value is the most likely and the probability falls to zero at the ends.

### B2.1 Draw the inputs and compute P

The inputs for a block like Reykjanes, with no field name attached:

| input | range | shape |
|---|---|---|
| $A$ | 2 to 7 km² | uniform |
| $h$ | 1.5 to 3 km | uniform |
| $T$ | 280 to 320 °C, most likely 300 | triangular |
| $R$ | 0.05 to 0.25, most likely 0.10 | triangular |
| $\eta$ | 0.10 to 0.12 | uniform |
| $\rho c$ | 2.4 to 2.8 MJ/(m³ K) | uniform |

$T_0$ = 30 °C, $F$ = 0.9 and 30 years are fixed. `rng.uniform(low, high, n)` and `rng.triangular(low, mode, high, n)` return `n` draws. Fill the gaps. Expected: P90 about 53 MWe, P50 about 110 MWe, P10 about 210 MWe (the exact values move a little with the random seed).

In [ ]:
rng = np.random.default_rng(1)      # the random number generator; the seed makes the draws repeatable
n = 100_000

A = rng.uniform(2e6, 7e6, n)                     # m2
h = rng.uniform(1500.0, 3000.0, n)               # m
T = rng.triangular(280.0, 300.0, 320.0, n)       # C
R = ___   # recovery factor, triangular 0.05 to 0.25 with mode 0.10
eta = rng.uniform(0.10, 0.12, n)
rho_c = rng.uniform(2.4e6, 2.8e6, n)             # J/(m3 K)

Q = stored_heat(A, h, T, T0=30.0, rho_c=rho_c)   # J, one value per draw
P_MWe = ___   # capacity in MWe for every draw, with the drawn R and eta

P90, P50, P10 = np.percentile(P_MWe, [10, 50, 90])   # exceeded with 90, 50 and 10 percent probability
print(f"P90 = {P90:.0f} MWe,  P50 = {P50:.0f} MWe,  P10 = {P10:.0f} MWe")
assert 45 < P90 < 60 and 100 < P50 < 120 and 190 < P10 < 230

### A. The histogram and the cumulative distribution

Left: how often each value of $P$ came up. Right: the share of the draws at or below each value, the cumulative distribution of slide 17. P90 is where the curve reaches 10 percent, P50 where it reaches 50, P10 where it reaches 90. The printed lines give the chance of at least 50, 100 and 150 MWe.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.hist(P_MWe, bins=80, color="#2a78d6", density=True)
for v, lab in ((P90, "P90"), (P50, "P50"), (P10, "P10")):
    ax1.axvline(v, color="k", ls="--"); ax1.text(v, 1.01, f"{lab} {v:.0f}", ha="center", va="bottom", transform=ax1.get_xaxis_transform())
ax1.set_xlabel("capacity P, MWe"); ax1.set_ylabel("probability density"); ax1.set_xlim(0, 400)
Ps = np.sort(P_MWe); share_below = 100 * np.arange(1, n + 1) / n
ax2.plot(Ps, share_below, color="#2a78d6", lw=2)
ax2.set_xlabel("capacity P, MWe"); ax2.set_ylabel("share of the draws at or below this value, %"); ax2.set_xlim(0, 400); ax2.set_ylim(0, 100)
plt.show()

def chance_of_at_least(P_target):
    '''Fraction of the draws at or above P_target (MWe), in percent.'''
    return 100 * np.mean(P_MWe >= P_target)
for target in (50, 100, 150):
    print(f"chance of at least {target} MWe: {chance_of_at_least(target):.0f} %")

### B2.2 Which input controls the range?

Two ways to ask. First, the **rank correlation** between each input and $P$ over all draws: an input with a large rank correlation moves $P$ a lot. `spearmanr(x, y)[0]` gives it. Second, the **tornado**: move one input from its low to its high value with every other input at its middle value. Fill the gap in the tornado. Expected: $R$ first, then $A$, then $h$; $T$, $\eta$ and $\rho c$ hardly matter.

In [ ]:
inputs = {"A": A, "h": h, "T": T, "R": R, "eta": eta, "rho_c": rho_c}
print("rank correlation with P:")
for name, x in inputs.items():
    print(f"  {name:6s} {spearmanr(x, P_MWe)[0]:5.2f}")

mid = dict(A=4.5e6, h=2250.0, T=300.0, R=0.10, eta=0.11, rho_c=2.6e6)
lo = dict(A=2e6, h=1500.0, T=280.0, R=0.05, eta=0.10, rho_c=2.4e6)
hi = dict(A=7e6, h=3000.0, T=320.0, R=0.25, eta=0.12, rho_c=2.8e6)

def P_of(d):
    '''Capacity in MWe for one set of inputs given as a dictionary.'''
    return capacity(stored_heat(d["A"], d["h"], d["T"], T0=30.0, rho_c=d["rho_c"]), R=d["R"], eta=d["eta"]) / 1e6

print(f"\nall inputs at the middle: {P_of(mid):.0f} MWe")
print("tornado, one input at a time from low to high:")
for name in mid:
    d_lo, d_hi = dict(mid), dict(mid)
    d_lo[name] = lo[name]
    d_hi[name] = hi[name]                        # the same dictionary with this one input at its high value
    print(f"  {name:6s} {P_of(d_lo):5.0f} to {P_of(d_hi):5.0f} MWe")

### A. Change one range

Suppose a new resistivity survey narrows the area to 3 to 5 km². Rerun with that range and see how much the P90 moves. Then try narrowing $R$ to 0.08 to 0.12 instead. Which survey was worth more?

In [ ]:
def monte_carlo(A_range=(2e6, 7e6), R_range=(0.05, 0.10, 0.25), n=100_000, seed=1):
    '''P90, P50, P10 in MWe for the block, with the area range (uniform) and the R range (triangular low, mode, high) as arguments.'''
    rng = np.random.default_rng(seed)
    A = rng.uniform(*A_range, n); h = rng.uniform(1500.0, 3000.0, n); T = rng.triangular(280.0, 300.0, 320.0, n)
    R = rng.triangular(*R_range, n); eta = rng.uniform(0.10, 0.12, n); rho_c = rng.uniform(2.4e6, 2.8e6, n)
    P = capacity(stored_heat(A, h, T, rho_c=rho_c), R=R, eta=eta) / 1e6
    return np.percentile(P, [10, 50, 90])

for label, kw in (("base case", {}), ("area 3 to 5 km2", dict(A_range=(3e6, 5e6))), ("R 0.08 to 0.12", dict(R_range=(0.08, 0.10, 0.12)))):
    P90, P50, P10 = monte_carlo(**kw)
    print(f"{label:18s} P90 {P90:4.0f}  P50 {P50:4.0f}  P10 {P10:4.0f} MWe")

## Part 3. Heat extraction from parallel fractures

The recovery factor was an assumption in Parts 1 and 2. Here it is computed from a physical model: water flows up $N$ parallel vertical fractures of height $H$, width $L$ and spacing $2x_E$ in rock that is impermeable between the fractures (Gringarten, Witherspoon and Ohnishi 1975). Heat reaches the fracture only by conduction across the rock block. By symmetry only one half block, from the fracture face at $x = 0$ to the midplane at $x = x_E$, has to be modelled.

Two equations, exactly as on the board.

Conduction in the rock, across the block (Fourier's law for the flux, a heat balance for the slab):

$$ q = -\lambda \frac{\partial T_R}{\partial x} \qquad\qquad \rho_R c_R \frac{\partial T_R}{\partial t} = -\frac{\partial q}{\partial x} $$

The water in the fracture, heated by both faces as it flows up ($m'$ is the mass flow per fracture per metre of width, $q_\mathrm{face}$ the heat flux from the rock into the fracture):

$$ c_w m' \frac{dT_w}{dz} = 2\, q_\mathrm{face} $$

$q$ heat flux (W/m²); $\lambda$ rock thermal conductivity (W/(m K)); $\rho_R c_R$ rock volumetric heat capacity (J/(m³ K)); $T_R$ rock temperature, $T_w$ water temperature (°C); $c_w$ specific heat of water (J/(kg K)); $m'$ mass flow per fracture per metre of width (kg/(m s)); $x$ distance from the fracture face, $z$ distance along the fracture (m).

### A. From the equations to a function

The construction is the one you used for pressure in Notebook 02, with slabs in place of rings.

Step 1. Cut the half block into $M$ slabs parallel to the fracture, thin near the face where the temperature changes fastest. Cut the fracture into $K$ segments along $z$.

Step 2. Fourier's law on each slab face (slide 29): the heat flux toward the fracture is the conductivity times the temperature difference between the two neighbouring slabs, divided by the distance between their centres.

Step 3. Heat balance for each slab (slide 30): the rate of temperature change is the flux in through the outer face minus the flux out through the inner face, divided by the volumetric heat capacity and the slab thickness.

Step 4. The water: within one segment the rock face temperature is taken constant, so the water temperature relaxes toward it along the segment, $T_w(z + dz) = T_\mathrm{face} + (T_w(z) - T_\mathrm{face})\, e^{-a\, dz}$ with $a = 2 \lambda / (x_1 c_w m')$, where $x_1$ is the distance from the face to the centre of the first slab. The heat the water gains in the segment is what the first slab loses through the face.

At the two ends of the grid: at the midplane no heat crosses (symmetry), and at the fracture face the flux is the heat taken by the water.

### A. The model in code

The code is complete; nothing to fill in. Find the two lines marked `step 2` and `step 3` and check them against the two sentences above. Everything else is organisation: cutting the block into slabs, moving the water up the fracture segment by segment, and a table that tells the solver which unknowns depend on each other.

In [ ]:
def make_model(m1, H, x_E, T_R0, T_in, lam_R=2.6, rho_R=2650.0, c_R=1000.0, c_w=4200.0, M=30, K=100, dx_min=0.02):
    '''Heat extraction from one fracture in a periodic array.
    m1 mass flow per fracture per metre of width (kg/(m s)); H fracture height (m); x_E half spacing (m);
    T_R0 initial rock temperature and T_in injection temperature (C); rock lam_R, thermal conductivity (W/(m K)), rho_R (kg/m3), c_R (J/(kg K)).
    Returns the slab centres x, the segment centres z, the function dTdt(t, T) for solve_ivp, and T_out_of(T).'''
    x_face = np.concatenate([[0.0], np.geomspace(dx_min, x_E, M)])   # M + 1 slab faces, thin slabs at the fracture
    x = 0.5 * (x_face[:-1] + x_face[1:])                              # slab centres, m
    dx = np.diff(x_face)                                              # slab thicknesses, m
    dz = H / K                                                        # segment length, m
    z = (np.arange(K) + 0.5) * dz
    a = 2 * lam_R / (x[0] * c_w * m1)                                   # 1/m, step 4

    def water_march(T_face):
        '''Water temperature at the K + 1 segment boundaries, from the inlet upward. T_face: rock face temperature of each segment.'''
        T_w = np.empty(K + 1); T_w[0] = T_in
        for k in range(K):
            T_w[k + 1] = T_face[k] + (T_w[k] - T_face[k]) * np.exp(-a * dz)
        return T_w

    def dTdt(t, T_flat):
        T = T_flat.reshape(K, M)                  # T[k, j]: rock slab j in segment k
        T_w = water_march(T[:, 0])
        q = np.zeros((K, M + 1))                  # heat flux toward the fracture on each slab face, W/m2
        q[:, 1:-1] = lam_R * (T[:, 1:] - T[:, :-1]) / (x[1:] - x[:-1])  # step 2, Fourier's law: conductivity times temperature difference over distance
        q[:, 0] = c_w * m1 * (T_w[1:] - T_w[:-1]) / (2 * dz)          # the face: heat taken by the water in the segment, per face
        q[:, -1] = 0.0                                                # the midplane: no heat crosses it
        dT_dt = (q[:, 1:] - q[:, :-1]) / (rho_R * c_R * dx)  # step 3, heat balance of every slab: (flux in minus flux out) over rho_R c_R dx
        return dT_dt.ravel()                                            # one long vector for the solver

    def T_out_of(T_flat):
        '''Outlet temperature for a given rock temperature field.'''
        return water_march(T_flat.reshape(K, M)[:, 0])[-1]

    # which unknowns influence which rates; lets the solver build its Jacobian cheaply (not physics, only speed)
    S = np.zeros((K * M, K * M), dtype=bool); idx = np.arange(K * M).reshape(K, M)
    for k in range(K):
        for j in range(M):
            S[idx[k, j], idx[k, max(j - 1, 0):j + 2]] = True
        S[idx[k, 0], idx[:k + 1, 0]] = True
    dTdt.jac_sparsity = S
    return x, z, dTdt, T_out_of


def run_fractures(m_total, N, H, L, spacing, T_R0, T_in, years=100.0, nt=201, **rock):
    '''Outlet temperature (C) and thermal power (W) against time (s) for N fractures of height H and width L (m) at the given spacing (m),
    with m_total kg/s in total. Also returns the final rock temperature field T[k, j] and the grids x, z.'''
    m1 = m_total / (N * L)                                            # kg/(m s)
    c_w = cw_mean(T_in, T_R0)                                         # J/(kg K), from IAPWS
    x, z, dTdt, T_out_of = make_model(m1, H, spacing / 2, T_R0, T_in, c_w=c_w, **rock)
    t_eval = np.linspace(0, years * YEAR, nt)
    sol = solve_ivp(dTdt, (0, years * YEAR), np.full(len(z) * len(x), T_R0), method="BDF", t_eval=t_eval,
                    rtol=1e-6, atol=1e-4, jac_sparsity=dTdt.jac_sparsity)
    T_out = np.array([T_out_of(sol.y[:, i]) for i in range(sol.y.shape[1])])
    power = c_w * m_total * (T_out - T_in)                            # W
    return sol.t, T_out, power, sol.y[:, -1].reshape(len(z), len(x)), x, z

# the example of Gringarten et al. 1975: 145 kg/s, 1 km by 1 km fractures, 300 C rock, injection at 40 C
t, T_out, power, T_field, x, z = run_fractures(145.0, N=10, H=1000.0, L=1000.0, spacing=160.0, T_R0=300.0, T_in=40.0)
print(f"10 fractures, 160 m spacing: T_out after 20 years {np.interp(20*YEAR, t, T_out):.0f} C, after 50 years {np.interp(50*YEAR, t, T_out):.0f} C")
assert abs(np.interp(20 * YEAR, t, T_out) - 297) < 4

The paper gives 297 °C after 20 years for this case with $c_w$ = 4186 J/(kg K). Here $c_w$ comes from IAPWS (4450 J/(kg K) between 40 and 300 °C) and the segments along the fracture are 10 m long, so the model gives a few degrees less. `K=200` halves the grid error at twice the run time.

### A. The rock after 30 years

The temperature field after 30 years, for one fracture in the array. The fracture is along the left edge.

In [ ]:
t, T_out, power, T_field, x, z = run_fractures(145.0, N=10, H=1000.0, L=1000.0, spacing=160.0, T_R0=300.0, T_in=40.0, years=30.0)
fig, ax = plt.subplots(figsize=(8, 5))
pc = ax.pcolormesh(x, z, T_field, shading="nearest", cmap="inferno", vmin=40, vmax=300)
fig.colorbar(pc, label="rock temperature, °C"); ax.set_xlabel("distance from the fracture face x, m"); ax.set_ylabel("distance along the fracture z, m")
ax.set_title("10 fractures 160 m apart, 145 kg/s, after 30 years", loc="left"); ax.grid(False)
plt.show()
print(f"cooled layer about sqrt(kappa t) = {np.sqrt(2.6/(2650*1000) * 30 * YEAR):.0f} m; the midplane is at {x[-1]:.0f} m")

### B3.1 One fracture, then ten at three spacings

Run the four cases of the lecture and plot $T_\mathrm{out}(t)$. From the paper calculation in class, $\ell$ = 8.5 m for one fracture and 85 m for ten, so $t_1 = \ell^2/\kappa$ = 2.3 years and 230 years; compare the curves with these numbers.

Fill the gap: the depletion time $t_2$ of slide 38, heat in place between the fractures over the thermal power at full temperature, $t_2 = \rho_R c_R V / (c_w \dot m)$. Expected: about 52 years for 40 m spacing (the lecture slide says 58 years with $c_w$ = 4200 J/(kg K); here $c_w$ comes from IAPWS and is 4450 J/(kg K) between 40 and 300 °C).

In [ ]:
cases = [("1 fracture", 1, 160.0), ("10 fractures, 40 m", 10, 40.0), ("10 fractures, 80 m", 10, 80.0), ("10 fractures, 160 m", 10, 160.0)]
rho_R, c_R, lam_R = 2650.0, 1000.0, 2.6                  # rock, assumed
m_total, H, L, T_R0, T_in = 145.0, 1000.0, 1000.0, 300.0, 40.0
c_w = cw_mean(T_in, T_R0)
results = {}
fig, ax = plt.subplots(figsize=(9, 5))
for label, N, spacing in cases:
    t, T_out, power, _, _, _ = run_fractures(m_total, N, H, L, spacing, T_R0, T_in, years=100.0)
    results[label] = (t, T_out, power, N, spacing)
    V = N * spacing * H * L                                # m3 of rock between the fractures
    t2 = ___   # depletion time t2 of slide 38, s
    ax.plot(t / YEAR, T_out, lw=2, label=label)
    print(f"{label:24s} T_out(30 yr) = {np.interp(30*YEAR, t, T_out):4.0f} C,  t2 = {t2/YEAR:4.0f} years")
ax.set_xlabel("time, years"); ax.set_ylabel("outlet temperature, °C"); ax.legend(); plt.show()

### A. Gringarten's scaling, checked with the model

The lecture says that $T_\mathrm{out}$ depends on time only through $t / t_1$ with $t_1 = \ell^2 / \kappa$ and $\ell = 2 \lambda H / (c_w m')$ (plus the spacing through $x_E / \ell$). Three single fractures with different height, flow and conductivity, and therefore different $t_1$, should fall on one curve when time is divided by $t_1$. The dashed line is the exact solution for one fracture, $T_{R0} - T_\mathrm{out} = (T_{R0} - T_\mathrm{in})\,\mathrm{erfc}(\tfrac{1}{2}\sqrt{t_1/t})$.

In [ ]:
from scipy.special import erfc
cases_scaling = [("H = 1000 m, 145 kg/s, lambda 2.6", 1000.0, 145.0, 2.6), ("H = 500 m, 145 kg/s, lambda 2.6", 500.0, 145.0, 2.6),
                 ("H = 1000 m, 50 kg/s, lambda 3.5", 1000.0, 50.0, 3.5)]
fig, ax = plt.subplots(figsize=(9, 5))
for label, H_, m_, lam_ in cases_scaling:
    c_w_ = cw_mean(T_in, T_R0)
    ell_ = 2 * lam_ / (c_w_ * m_ / (1 * L)) * H_          # m, with N = 1
    t1_ = ell_**2 / (lam_ / (rho_R * c_R))                # s
    t_, T_out_, _, _, _, _ = run_fractures(m_, 1, H_, L, 2000.0, T_R0, T_in, years=10 * t1_ / YEAR, nt=201, lam_R=lam_)
    ax.plot(t_ / t1_, T_out_, lw=2, label=f"{label}: t1 = {t1_/YEAR:.1f} years")
tD = np.linspace(0.01, 10, 300)
ax.plot(tD, T_R0 - (T_R0 - T_in) * erfc(0.5 / np.sqrt(tD)), "k--", lw=1.5, label="exact solution for one fracture")
ax.set_xlabel("time / t1"); ax.set_ylabel("outlet temperature, °C"); ax.set_xlim(0, 10); ax.legend(); plt.show()

### A. Does the energy balance close?

The heat the water has carried out must equal the heat the rock has lost. The model is built from heat balances, so this should hold to the accuracy of the time stepping. The rock half block is one side of one fracture per metre of width; two half blocks per fracture, $N$ fractures, $L$ metres wide.

In [ ]:
t, T_out, power, T_field, x, z = run_fractures(m_total, 10, H, L, 40.0, T_R0, T_in, years=30.0)
dx = np.diff(np.concatenate([[0.0], np.geomspace(0.02, 20.0, 30)]))   # the slab thicknesses used in make_model for x_E = 20 m
heat_out = np.trapezoid(power, t)                                        # J, area under the power curve
heat_lost = 2 * 10 * L * np.sum(rho_R * c_R * (T_R0 - T_field) * dx[None, :] * (H / 100))   # J, two half blocks, 10 fractures, L wide
print(f"heat carried out by the water {heat_out:.3e} J,  heat lost by the rock {heat_lost:.3e} J,  ratio {heat_out/heat_lost:.3f}")

### A. Play with the numbers

The cell below runs one case and prints the outlet temperature after 10 and 30 years. Change one number at a time. Suggestions: double the flow; halve the spacing; make the fractures 500 m high instead of 1000 m; put the injection at 100 °C. Which change moves $T_\mathrm{out}$ after 30 years the most, and why?

In [ ]:
m_dot, N, H, L, spacing, T_R0, T_in = 145.0, 10, 1000.0, 1000.0, 160.0, 300.0, 40.0      # change these
t, T_out, power, _, _, _ = run_fractures(m_dot, N, H, L, spacing, T_R0, T_in, years=30.0, nt=61)
print(f"{N} fractures, {spacing:.0f} m apart, {m_dot:.0f} kg/s, H = {H:.0f} m: T_out after 10 years {np.interp(10*YEAR, t, T_out):.0f} C, after 30 years {np.interp(30*YEAR, t, T_out):.0f} C")

## Part 4. The recovery factor computed

$$ R(t) = \frac{\int_0^t c_w \dot m\,(T_\mathrm{out} - T_\mathrm{in})\, dt'}{\rho_R c_R V_B (T_{R0} - T_\mathrm{in})} $$

The numerator is the area under the thermal power curve, the heat the water has carried out by time $t$. The denominator is the heat in place in the block $V_B$, the block the developer counts as the resource, counted from $T_{R0}$ down to $T_\mathrm{in}$, as in Part 1. This is the same recovery factor that Part 2 drew from a triangular distribution.

$V_B$ is the same block for every case, whatever $N$ and the spacing. Here it is 1 km by 1 km by 1.6 km across the fractures, the block that holds ten fractures 160 m apart: $V_B = 1.6 \times 10^9$ m³. Ten fractures 40 m apart occupy the middle 400 m of it and the rest of the block is not touched; one fracture cools a few tens of metres on each side. If the heat in place were counted only between the fractures, one fracture would get a higher $R$ than ten, because its volume is smaller. That is the wrong result for a developer who has the whole block.

### B4.1 R against time for the four cases

While the water leaves at $T_{R0}$ the heat produced grows as $c_w \dot m (T_{R0} - T_\mathrm{in})\, t$, so $R = t / t_B$ with $t_B = \rho_R c_R V_B / (c_w \dot m)$, about 210 years here (the lecture slide says 232 years with $c_w$ = 4200 J/(kg K); here $c_w$ is 4450 J/(kg K) from IAPWS). All three ten-fracture cases give $R \approx 30/210 = 0.14$ after 30 years, and one fracture, whose outlet has cooled since its first year, much less.

`np.trapezoid(y, x)` integrates. The cumulative integral at every time is `np.concatenate([[0], np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(x))])`. Fill the gap.

In [ ]:
V_B = 1000.0 * 1000.0 * 1600.0                           # m3, the block every case is credited with

def recovery(t, power, V_B, T_R0, T_in):
    '''Recovery factor against time: cumulative heat produced over the heat in place in the block V_B.'''
    heat_produced = np.concatenate([[0.0], np.cumsum(0.5 * (power[1:] + power[:-1]) * np.diff(t))])   # J
    heat_in_place = ___   # heat in place in the block, J
    return heat_produced / heat_in_place

t_B = rho_R * c_R * V_B / (c_w * m_total)
print(f"t_B = {t_B/YEAR:.0f} years, so R = t/t_B = {30*YEAR/t_B:.3f} after 30 years while the outlet holds")
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot([0, 100], [0, 100 * YEAR / t_B], color="grey", ls="--", label="t / t_B")
for label, (t, T_out, power, N, spacing) in results.items():
    Rt = recovery(t, power, V_B, T_R0, T_in)
    ax.plot(t / YEAR, Rt, lw=2, label=label)
    print(f"{label:24s} R after 30 years = {np.interp(30*YEAR, t, Rt):.3f}")
ax.axvline(30, color="grey", ls=":"); ax.set_xlabel("time, years"); ax.set_ylabel("recovery factor R"); ax.set_ylim(0, 0.5); ax.legend(); plt.show()

### A. R after 30 years against the number of fractures, three flow rates

The same block, $N$ fractures spread evenly across it (spacing $1600/N$ m), the same rock. Three flow rates through the block. About 20 runs, under a minute.

In [ ]:
Ns = np.array([1, 2, 3, 5, 10, 20, 50])
flows = (50.0, 145.0, 400.0)
fig, ax = plt.subplots(figsize=(9, 5))
R30 = {}
for m_dot in flows:
    R30[m_dot] = []
    for N in Ns:
        t, T_out, power, _, _, _ = run_fractures(m_dot, int(N), H, L, 1600.0 / N, T_R0, T_in, years=30.0, nt=61)
        R30[m_dot].append(recovery(t, power, V_B, T_R0, T_in)[-1])
    ax.plot(Ns, R30[m_dot], "-o", lw=2, label=f"{m_dot:.0f} kg/s")
ax.set_xscale("log"); ax.set_xticks(Ns); ax.set_xticklabels([str(n) for n in Ns])
ax.set_xlabel("number of fractures N across the block"); ax.set_ylabel("recovery factor after 30 years"); ax.set_ylim(0, 0.45); ax.legend(title="flow through the block"); plt.show()

Read the plot: $R$ rises with $N$ until the outlet holds $T_{R0}$ for the whole 30 years, and from there on it is $t / t_B$, set by the flow through the block alone. More flow raises $R$ but needs more fractures to hold the temperature. $N$ and the spacing decide how long the outlet holds; while it holds, the flow alone decides $R$.



## Part 5. The same model for a doublet (Fervo type)

Turn the picture on its side. Two horizontal laterals of length $L_\mathrm{lat}$ are drilled $D$ apart and connected by $N$ vertical fractures, spaced $L_\mathrm{lat}/N$. Water flows horizontally through each fracture from the injector to the producer, so the flow length of the model is $H = D$, and the fracture width $L$ of the model is the fracture height, up and down from the wells. The block counted as the resource of the doublet is $V_B = L_\mathrm{lat} \cdot D \cdot L$.

Fervo's Cape project (Utah): from Fercho et al. (2025, Stanford Geothermal Workshop), about 100 kg/s per production well, rock at about 220 °C, reinjection at 80 °C, a stimulated reservoir volume about 1 km thick, laterals of about 1.5 km, and a fracture surface area of about 30 million square feet (2.8 million m²) per producer. Their numerical forecast: the production temperature falls from 220 to 180 °C over 30 years, and 59 percent of the heat in the stimulated volume is recovered. The well separation is not given as one number; 150 m is used here.

The model counts only the rock between the laterals and within the fracture height. Heat from above and below, and from beyond the ends of the laterals, is ignored; in 30 years conduction reaches about 30 m, so the real block is somewhat larger and the real decline somewhat slower than the model says.

`run_doublet` below calls `run_fractures` with the doublet names for the inputs. Nothing to fill in; this is the tool for Assignment 3, part 2.

In [ ]:
def run_doublet(L_lat, D, L, N, m_dot, T_R0, T_in, years=30.0, nt=121, **rock):
    '''Doublet: laterals of length L_lat (m) a distance D (m) apart, N fractures of height L (m) across both, m_dot (kg/s) through all.
    Returns t (s), T_out (C), power (W), R(t) counted in the block L_lat * D * L, and t_B (s).'''
    t, T_out, power, _, _, _ = run_fractures(m_dot, N, D, L, L_lat / N, T_R0, T_in, years=years, nt=nt, **rock)
    V_B = L_lat * D * L
    c_w = cw_mean(T_in, T_R0)
    t_B = rho_R * c_R * V_B / (c_w * m_dot)
    return t, T_out, power, recovery(t, power, V_B, T_R0, T_in), t_B

rho_R, c_R = 2650.0, 1000.0
fig, ax = plt.subplots(figsize=(9, 5))
for N in (10, 30, 100):
    t, T_out, power, Rt, t_B = run_doublet(1500.0, 150.0, 1000.0, N, 100.0, 220.0, 80.0)
    ax.plot(t / YEAR, T_out, lw=2, label=f"{N} fractures, {1500/N:.0f} m apart")
    print(f"{N:4d} fractures: T_out after 10 years {np.interp(10*YEAR, t, T_out):4.0f} C, after 30 years {np.interp(30*YEAR, t, T_out):4.0f} C, R after 30 years {Rt[-1]:.2f}")
print(f"t_B = {t_B/YEAR:.0f} years for this block")
ax.plot([0, 30], [217, 181], "k--", lw=1.5, label="Fervo's numerical forecast (Fercho et al. 2025)")
ax.set_xlabel("time, years"); ax.set_ylabel("outlet temperature, °C"); ax.set_ylim(70, 230); ax.legend(); plt.show()

Read the plot against the three numbers. The block holds $t_B$ = 45 years of heat at full power, so within 30 years the question is only how long the outlet holds, that is $t_1$. With 30 fractures, $\ell$ = 56 m and $t_1$ = 100 years, but the half spacing is 25 m, so the cooled zones meet after about 20 years and the outlet is at 177 °C after 30 years with $R$ = 0.60, close to Fervo's own forecast. With 10 fractures, $\ell$ = 19 m and $t_1$ = 11 years, and the outlet is at 127 °C after 30 years. The number that matters most is the number of fractures that actually carry flow.

## Possible extensions

1. **Usable recovery.** A flash plant cannot use water below about 180 °C. Define the usable recovery factor as the heat produced while $T_\mathrm{out}$ is above a cut-off temperature, divided by the heat in place in the block. Compute it for the four cases. The single fracture case changes the most.
2. **Flow rate.** Keep 10 fractures at 160 m and run flows from 50 to 800 kg/s. Plot $T_\mathrm{out}$ after 30 years and $R$ after 30 years against the flow rate. $\ell$ falls with the flow per fracture, so there is a flow above which the temperature does not hold for 30 years, and above it $R$ grows more slowly than $t / t_B$.
3. **A natural reservoir.** The model was written for created fractures and injected water. For a natural high-temperature reservoir the fractures are natural and the cold water is the recharge drawn in by production. Which inputs would you change, and which assumption of the model is the weakest in that case?
4. **The pore water.** The model counts only the rock. Add the pore water of Part 1 to the heat in place and see how $R$ changes.
5. **Back to the Monte Carlo of Part 2.** Suppose the block of Part 2 is produced at 145 kg/s and the stimulation creates between 2 and 20 fractures across it, most likely 5. Take $R$ after 30 years at $N$ = 2, 5 and 20 from the Part 4 plot as the low, mode and high of the triangular $R$ in `monte_carlo` (`R_range=(low, mode, high)`) and see how P90, P50 and P10 move.

## Assignment 3, part 2: heat extraction from fractured rock

Assignment 3 has two parts. Part 1 is notebook 02 (well tests, Lecture 6). Part 2 is this one. Both parts are handed in together; the deadline is on Canvas.


A Cape-type doublet in rock at $T_{R0}$ = 220 °C, produced at $\dot m$ = 100 kg/s with injection at $T_\mathrm{in}$ = 80 °C. Laterals $L_\mathrm{lat}$ = 1.5 km, $D$ = 150 m apart, fractures $L$ = 1 km high. Rock: $\lambda$ = 2.6 W/(m K), $\rho_R$ = 2650 kg/m³, $c_R$ = 1000 J/(kg K). Take $c_w$ = 4200 J/(kg K) for the hand calculations. Use `run_doublet` from Part 5 for the computed parts.

1. By hand: $m'$, $\ell$ and $t_1 = \ell^2/\kappa$ for $N$ = 10 and $N$ = 100. Define every symbol with its unit directly after the equation in which it first appears. (25 %)
2. By hand: $V_B$ and $t_B = \rho_R c_R V_B/(c_w \dot m)$. Can the heat in the block run out within 30 years? (15 %)
3. With `run_doublet`: $N$ = 10, 30 and 100 (one figure, three curves, 30 years). Report $T_\mathrm{out}$ after 10 and 30 years and $R$ after 30 years for each. How many fractures are needed to keep the outlet above 180 °C for 30 years? Compare with Fervo's forecast of 220 to 180 °C and $R$ = 0.59 (Fercho et al. 2025). (35 %)
4. Repeat $N$ = 30 with fractures only 500 m high ($L$ = 500 m). What changes, and why? (10 %)
5. In at most ten sentences: two things the model leaves out for a doublet, and for each the direction in which it moves the real outlet temperature. (15 %)

Hand in a short written report (PDF) with the equations and your answers, plus this notebook as .ipynb with the cells of Parts 1 to 4 completed and run and your assignment cells added after Part 5, together with notebook 02 for part 1.

## Reflection

Before you close the notebook, write three lines: the one number from today you will remember, the one assumption of the fracture model you trust least, and one question you still have.